Does cluster analysis make sense here?

Clustering is unsupervised, meaning it ignores my labels. Since I already have ground-truth classes:

- I can use clustering as a sanity check → “Do my data naturally form 7 groups?”

- I can evaluate how well unsupervised clusters line up with my known classes (using metrics like Adjusted Rand Index, Normalized Mutual Information, etc). --> This is what I am doing here

- Discover subclusters or overlaps that labels don’t capture (e.g., mislabeled images or natural subclasses).

Images are high dimensional. Eg, 224x224x3 ~ 150K dimensions. So we dont do clustering on this directly. But we get feature embeddings and do the clustering on this low dimensional data.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.manifold import TSNE


import torch
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

In [ ]:
# 1. Load dataset
data_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "processed", "train")


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

In [ ]:
class_names = dataset.classes
y_true = [y for _, y in dataset.samples]
print(class_names)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Upto this point is common code for both resnet model pretrained on ImageNet dataset and my Resnet model fine tuned on the wafermap dataset 

So ensure to run the code upto this point no matter what

In [ ]:
# 2. Feature extractor - low dimensional embeddings
model = models.resnet18(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1]) # drop last FC
model.to(device)    
model.eval()

In [ ]:
features = []
labels = []

with torch.no_grad():
    for x, y in dataloader:
        x = x.to(device)
        out = model(x) # (batch_size, 512, 1, 1)
        out = out.view(out.size(0), -1) # flatten (batch_size, 512)
        features.append(out.cpu().numpy()) # list of (batch_size, 512) size 2D numpy arrays
        labels.extend(y.numpy())


features = np.vstack(features) # (num_samples, 512)
labels = np.array(labels)

In [ ]:
print(features.shape, labels.shape)

In [ ]:
# 3. Dimensionality reduction
pca = PCA(n_components=50)
X_pca = pca.fit_transform(features)
print(X_pca.shape)

In [ ]:
# 4. Clustering
kmeans = KMeans(n_clusters=7, random_state=0)
cluster_labels = kmeans.fit_predict(X_pca)
print(cluster_labels.shape)

In [ ]:
# 5. Evaluation
ari = adjusted_rand_score(labels, cluster_labels)
nmi = normalized_mutual_info_score(labels, cluster_labels)
print(f"ARI: {ari:.3f}, NMI: {nmi:.3f}")

The below is a little bit of a tangent which shows that the labels dont need to match exactly. As long as the proportions are correct, these cluster metrics will be high.

In [ ]:
actual_labels = np.array([0,0,1,2,1,0,1,2,2,0])
predicted_labels = np.array([1,1,2,0,2,1,2,0,0,1])
#predicted_labels = np.array([1,1,2,0,2,1,1,1,1,1])

print(adjusted_rand_score(actual_labels, predicted_labels))
print(normalized_mutual_info_score(actual_labels, predicted_labels))

In [ ]:
# 6. Visualization
tsne = TSNE(n_components=2, random_state=0, perplexity=30)
X_tsne = tsne.fit_transform(X_pca)

plt.figure(figsize=(12, 5))

# Plot by true labels
plt.subplot(1, 2, 1)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap="tab10", s=10)
plt.title("True Classes")

# Plot by clusters
plt.subplot(1, 2, 2)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap="tab10", s=10)
plt.title("KMeans Clusters")

plt.show()

Do the same as above but using the fine tuned (on the wafermap dataset) resnet152 model

In [ ]:
import os
import torch

from nanodefectnet.utils.config_utils import get_inference_config
from nanodefectnet.core.inference_classification_model import get_trained_model_path, load_model
from nanodefectnet.utils.constants import ModelType

In [ ]:
inference_config_file_path=os.path.join(os.path.dirname(os.getcwd()), "configs/inference/infer.yaml")
model_name = ModelType.RESNET152.value

In [ ]:
inference_config = get_inference_config(inference_config_file_path)

model_path = os.path.join(os.path.dirname(os.getcwd()), get_trained_model_path(model_name, inference_config))
resnet_model = load_model(model_path, model_type=model_name)

image_embedding_extractor = torch.nn.Sequential(*list(resnet_model.model.model.children())[:-1]) # drop last FC
image_embedding_extractor.eval()


In [ ]:
print(resnet_model.model.model)

# yea its deeply nested. Welcome to fine tuning pytorch models

# Resnet152ModelInference ---> Resnet152Model ---> model (this is what we need)

I see the `AdaptiveAvgPool((1,1))` in my model. So thats not chopped off.

In [ ]:
print(image_embedding_extractor)

In [ ]:
# How to check if the model is on the CPU or GPU?

print(next(image_embedding_extractor.parameters()).device)

In [ ]:
image_embedding_extractor.to(device)

In [ ]:
features = []
labels = []

with torch.no_grad():
    for x, y in dataloader:
        x = x.to(device)
        out = image_embedding_extractor(x)
        out = out.view(out.size(0), -1) # flatten
        features.append(out.cpu().numpy())
        labels.extend(y.numpy())


features = np.vstack(features)
labels = np.array(labels)

In [ ]:
print(features.shape, labels.shape)

In [ ]:
# 3. Dimensionality reduction
pca = PCA(n_components=50)
X_pca = pca.fit_transform(features)
print(X_pca.shape)

In [ ]:
# 4. Clustering
kmeans = KMeans(n_clusters=7, random_state=0)
cluster_labels = kmeans.fit_predict(X_pca)
print(cluster_labels.shape)

In [ ]:
# 5. Evaluation
ari = adjusted_rand_score(labels, cluster_labels)
nmi = normalized_mutual_info_score(labels, cluster_labels)
print(f"ARI: {ari:.3f}, NMI: {nmi:.3f}")

In [ ]:
# 6. Visualization
tsne = TSNE(n_components=2, random_state=0, perplexity=30)
X_tsne = tsne.fit_transform(X_pca)

plt.figure(figsize=(12, 5))

# Plot by true labels
plt.subplot(1, 2, 1)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap="tab10", s=10)
plt.title("True Classes")

# Plot by clusters
plt.subplot(1, 2, 2)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=cluster_labels, cmap="tab10", s=10)
plt.title("KMeans Clusters")

plt.show()

Not that great results it would seem